# IBL Brain Wide map- Dataset conversion

Data from Meshulam et al. (2025)¹ , a multi-lab dataset studying the neural basis of decision-making using neuropixel probes. This data set contains other information gathered during the task: sensory stimuli presented to the mouse; mouse decisions and response times; and mouse pose information from video recordings and DeepLabCut analysis.
 
- Link to entire dataset: https://neurosift.app/dandiset/000409
- Link to one subject session (downloaded here) https://neurosift.app/nwb?url=https://api.dandiarchive.org/api/assets/00cab48e-f9e2-497e-b1a8-cc19a019ee07/download/&dandisetId=000409 for subject CSHL045 from Churchland lab.


The code below shows how one can download and convert a single session `nwb` file (behavioural/pose data only) into the `TrialTree` format, as well as download some external videos linked to the file. 

**Note**: A significant amount of the code (in `ethograph.utils.nwb`) is dedicated to automatically identifying to identifying the video download links associated with this session usinig NWB `external_file` attribute, and then identifying which `nwb` PoseEstimationSeries data (see https://github.com/rly/ndx-pose) belong to which idea (e.g. Pose from CameraLeft to .mp4 of CameraLeft). Maybe this will be made easier in the future! See Git issue [here](https://github.com/int-brain-lab/iblenv/issues/432).

Note, that poes estimation data has to be `ndx-pose>v0.2.0`.


---
Meshulam, L., Angelaki, D., Benson, B., Benson, J., Birman, D., Arlandis, J., Bonacchi, N., Bougrova, K., Bruijns, S. A., Carandini, M., Catarino, J. A., Chapuis, G. A., Churchland, A. K., Dan, Y., Davatolhagh, F., Dayan, P., DeWitt, E. E., Engel, T. A., Fabbri, M., … Witten, I. B. (2025). A brain-wide map of neural activity during complex behaviour. Nature, 645(8079), 177–191. https://doi.org/10.1038/s41586-025-09235-0



In [ ]:
%load_ext autoreload
%autoreload 2
import remfile
import h5py
import pynwb
from dandi.dandiapi import DandiAPIClient
from ethograph.utils.nwb import load_nwb_session, find_video_assets, match_dandi_videos
from ethograph.utils.label_intervals import NWBLabelConverter


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
dandiset_id = "000409" 
asset_id = "773516a9-bd20-4b46-adad-2a1d5772be5d" # 
with DandiAPIClient() as client:
    asset = client.get_dandiset(dandiset_id).get_asset(asset_id)
    url = asset.get_content_url(follow_redirects=1, strip_query=True)


rf = remfile.File(url)
h5_file = h5py.File(rf, "r")
io = pynwb.NWBHDF5IO(file=h5_file, mode="r", load_namespaces=True)


nwb_file = io.read()

2026-03-06 21:06:14.361 | WARNING  | hdmf.spec.namespace:warn_for_ignored_namespaces:620 - c:\Users\aksel\anaconda3\envs\ethograph\Lib\site-packages\hdmf\spec\namespace.py:590: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
hdmf-common - cached version: 1.9.0, loaded version: 1.8.0
hdmf-experimental - cached version: 0.6.0, loaded version: 0.5.0
Please update to the latest package versions.
  self.warn_for_ignored_namespaces(ignored_namespaces)



In [21]:
registry = get_pose_camera_registry(nwb_file)
dandi_assets = find_video_assets(dandiset_id, nwb_file, asset_id)
match_dandi_videos(registry, dandi_assets, asset_id)

print(f"Video & Pose registry: {registry}")

Matched by camera key: LeftCamera -> https://api.dandiarchive.org/api/assets/e2d4561a-e0ec-4b89-aad0-acc246e9aa46/download/
Matched by camera key: RightCamera -> https://api.dandiarchive.org/api/assets/e8d5d62d-fb20-44e8-b8b4-a210f3953580/download/
Video & Pose registry: {'LeftCamera': {'processing_module_key': 'pose_estimation', 'pose_interface_key': 'LeftCamera', 'video_url': 'https://api.dandiarchive.org/api/assets/e2d4561a-e0ec-4b89-aad0-acc246e9aa46/download/'}, 'RightCamera': {'processing_module_key': 'pose_estimation', 'pose_interface_key': 'RightCamera', 'video_url': 'https://api.dandiarchive.org/api/assets/e8d5d62d-fb20-44e8-b8b4-a210f3953580/download/'}}


In [ ]:
dt, trials_df = load_nwb_session(nwb_file, registry, load_nwb_session=True)
label_converter = NWBLabelConverter()
label_dt = label_converter.from_nwb(nwb_file, trials_df)

### Video and pose don't data don't match?

In [8]:
import cv2
from typing import Any
from pathlib import Path
from dandi.dandiapi import DandiAPIClient


def find_video_assets(
    dandiset_id: str,
    nwb: Any,
) -> list[tuple[str, str]]:
    session_id = getattr(nwb, "session_id", None)
    identifier = getattr(nwb, "identifier", None)
    subject_id = getattr(getattr(nwb, "subject", None), "subject_id", None)

    search_terms = [t for t in [session_id, identifier[:8] if identifier else None, subject_id] if t]
    if not search_terms:
        return []

    with DandiAPIClient() as client:
        dandiset = client.get_dandiset(dandiset_id)
        video_assets = [
            (Path(asset.path).stem, f"https://api.dandiarchive.org/api/assets/{asset.identifier}/download/")
            for asset in dandiset.get_assets()
            if asset.path.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
            and any(term in asset.path for term in search_terms)
        ]

    return video_assets

dandiset_id = "000231" # 000409
with DandiAPIClient() as client:
    asset = client.get_dandiset(dandiset_id).get_asset(
        "ead24fb9-0d16-414a-9f40-0c8a7bd5ee2c"
    )
    url = asset.get_content_url(follow_redirects=1, strip_query=True)

# remfile — better h5py compatibility, recommended for movement/behavior converters
rf = remfile.File(url)
h5_file = h5py.File(rf, "r")
io = pynwb.NWBHDF5IO(file=h5_file, mode="r", load_namespaces=True)


nwb_file = io.read()


video_assets = find_video_assets(dandiset_id, nwb_file)



2026-03-06 08:36:32.806 | WARNING  | hdmf.spec.namespace:warn_for_ignored_namespaces:620 - c:\Users\aksel\anaconda3\envs\ethograph\Lib\site-packages\hdmf\spec\namespace.py:590: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
ndx-pose - cached version: 0.1.1, loaded version: 0.2.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)

